# HW2 - Classification

## Problem 2: Model and Analyze a Real Dataset

Goal: predict whether a flight's **departure will be delayed**, using `DelayorNot_processed.csv`.

Features used:
- `FLIGHT_DIRECTION`: general compass direction from origin state to destination state (0 = same state, 1 = Northwest, 2 = Southwest, 3 = Northeast, 4 = Southeast)
- `CRS_DEP_TIME`: scheduled departure time (continuous)
- `DAY_OF_MONTH`: day of the month (continuous)

`CRS_DEP_TIME` and `DAY_OF_MONTH` have much bigger numbers than the 0/1 dummy columns, so I scaled them with `StandardScaler` to bring them down to a similar size to `FLIGHT_DIRECTION`. This step ensures KNN's distance calculation doesn't just compare the time/day values while ignoring the other features.

### Import the Data

In [ ]:
import pandas as pd

df = pd.read_csv("DelayorNot_processed_all.csv")
df.head()


### Data Processing

Build the binary target from `DEP_DELAY_NEW`, one-hot encode `FLIGHT_DIRECTION`, split into train/test sets (stratified), then standardize the two continuous columns.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

y = (df["DEP_DELAY_NEW"] > 0).astype(int)
X = df[["FLIGHT_DIRECTION", "CRS_DEP_TIME", "DAY_OF_MONTH"]]
X = pd.get_dummies(X, columns=["FLIGHT_DIRECTION"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

# standardize the continuous columns (KNN's distance metric needs this;
# it doesn't hurt logistic regression either)
continuous_cols = ["CRS_DEP_TIME", "DAY_OF_MONTH"]
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test_scaled[continuous_cols] = scaler.transform(X_test[continuous_cols])

print("train size:", len(X_train_scaled), " test size:", len(X_test_scaled))
print("delay rate:", y.mean())


### Fit a KNN Classifier

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=15)
knn.fit(X_train_scaled, y_train)


### Fit a Second Classifier

Logistic Regression.

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train_scaled, y_train)


### Model Analysis

Test error rate and AUC for both models.

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score

models = {"KNN (k=15)": knn, "Logistic Regression": logreg}
for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    error_rate = 1 - accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    print(f"{name}: test error rate = {error_rate:.4f}, AUC = {auc:.4f}")


### KNN Test Error vs. K

In [ ]:
import matplotlib.pyplot as plt

k_values = [1, 5, 10, 15, 20, 50, 100, 200, 500, 1000]
test_errors = []
for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    err = 1 - accuracy_score(y_test, model.predict(X_test_scaled))
    test_errors.append(err)

plt.plot(k_values, test_errors, marker="o")
plt.xlabel("K")
plt.ylabel("Test error rate")
plt.title("KNN test error vs. K")
plt.show()

# find the K with the lowest test error among the K values tried above
best_idx = test_errors.index(min(test_errors))
best_k = k_values[best_idx]
print(f"best K = {best_k}, test error = {test_errors[best_idx]:.4f}")


## Conclusion

Both models' AUC scores are higher than the 0.5 "random guessing" baseline, showing that `FLIGHT_DIRECTION`, `CRS_DEP_TIME`, and `DAY_OF_MONTH` do carry real predictive signal about delays. However, since the AUC is only somewhat above 0.5, this signal isn't very strong, suggesting these features aren't actually that strongly associated with flight departure delays. If we compute the prior probability of delay for each feature value, we find that the delay probabilities are all very close to each other. KNN slightly outperforms Logistic Regression on both metrics, which suggests the relationship between these features and delay may be somewhat non-linear. The best choice of K is approximately 200.